# 3600. Maximize Spanning Tree Stability with Upgrades

You are given an integer n, representing n nodes numbered from 0 to n - 1 and a list of edges, where edges[i] = [ui, vi, si, musti]:

ui and vi indicates an undirected edge between nodes ui and vi.
si is the strength of the edge.
musti is an integer (0 or 1). If musti == 1, the edge must be included in the spanning tree. These edges cannot be upgraded.
You are also given an integer k, the maximum number of upgrades you can perform. Each upgrade doubles the strength of an edge, and each eligible edge (with musti == 0) can be upgraded at most once.

The stability of a spanning tree is defined as the minimum strength score among all edges included in it.

Return the maximum possible stability of any valid spanning tree. If it is impossible to connect all nodes, return -1.

Note: A spanning tree of a graph with n nodes is a subset of the edges that connects all nodes together (i.e. the graph is connected) without forming any cycles, and uses exactly n - 1 edges.

Example 1:
```
Input: n = 3, edges = [[0,1,2,1],[1,2,3,0]], k = 1

Output: 2

Explanation:

Edge [0,1] with strength = 2 must be included in the spanning tree.
Edge [1,2] is optional and can be upgraded from 3 to 6 using one upgrade.
The resulting spanning tree includes these two edges with strengths 2 and 6.
The minimum strength in the spanning tree is 2, which is the maximum possible stability.
```

Example 2:
```
Input: n = 3, edges = [[0,1,4,0],[1,2,3,0],[0,2,1,0]], k = 2

Output: 6

Explanation:

Since all edges are optional and up to k = 2 upgrades are allowed.
Upgrade edges [0,1] from 4 to 8 and [1,2] from 3 to 6.
The resulting spanning tree includes these two edges with strengths 8 and 6.
The minimum strength in the tree is 6, which is the maximum possible stability.
```
Example 3:
```
Input: n = 3, edges = [[0,1,1,1],[1,2,1,1],[2,0,1,1]], k = 0

Output: -1

Explanation:

All edges are mandatory and form a cycle, which violates the spanning tree property of acyclicity. Thus, the answer is -1.
```

Constraints:
```
2 <= n <= 105
1 <= edges.length <= 105
edges[i] = [ui, vi, si, musti]
0 <= ui, vi < n
ui != vi
1 <= si <= 105
musti is either 0 or 1.
0 <= k <= n
There are no duplicate edges.
```

In [ ]:
from typing import List

class Solution:
    def maxStability(self, n: int, edges: List[List[int]], k: int) -> int:

        class UnionFind:
            def __init__(self, n):
                self.parent = list(range(n))
                self.rank = [0] * n

            def find(self, x):
                while self.parent[x] != x:
                    self.parent[x] = self.parent[self.parent[x]]
                    x = self.parent[x]
                return x

            def union(self, x, y):
                px, py = self.find(x), self.find(y)
                if px == py:
                    return False
                if self.rank[px] < self.rank[py]:
                    px, py = py, px
                self.parent[py] = px
                if self.rank[px] == self.rank[py]:
                    self.rank[px] += 1
                return True

        # 필수 간선들이 사이클을 형성하는지 사전 체크
        uf = UnionFind(n)
        for u, v, s, must in edges:
            if must == 1 and not uf.union(u, v):
                return -1

        def feasible(mid):
            uf = UnionFind(n)

            # 필수 간선: 강도 < mid 이면 불가
            for u, v, s, must in edges:
                if must == 1:
                    if s < mid:
                        return False
                    uf.union(u, v)

            # Type A: 선택 간선, 업그레이드 불필요 (s >= mid)
            for u, v, s, must in edges:
                if must == 0 and s >= mid:
                    uf.union(u, v)

            # Type B: 선택 간선, 업그레이드 1회 필요 (s < mid <= s*2)
            upgrades = 0
            for u, v, s, must in edges:
                if must == 0 and s < mid <= s * 2:
                    if upgrades < k and uf.union(u, v):
                        upgrades += 1

            # 전체 연결 확인
            root = uf.find(0)
            return all(uf.find(i) == root for i in range(n))

        # 어떤 안정성 값도 달성 불가 (연결 자체 불가)
        if not feasible(1):
            return -1

        # 후보값: 각 간선의 원래 강도 + 선택 간선의 2배 강도
        candidates = sorted(
            {s for u, v, s, must in edges} |
            {s * 2 for u, v, s, must in edges if must == 0}
        )

        # 이진 탐색으로 최대 안정성 탐색
        lo, hi = 0, len(candidates) - 1
        ans = 1

        while lo <= hi:
            mid_idx = (lo + hi) // 2
            if feasible(candidates[mid_idx]):
                ans = candidates[mid_idx]
                lo = mid_idx + 1
            else:
                hi = mid_idx - 1

        return ans


# 테스트
sol = Solution()
print(sol.maxStability(3, [[0,1,2,1],[1,2,3,0]], 1))       # 2
print(sol.maxStability(3, [[0,1,4,0],[1,2,3,0],[0,2,1,0]], 2))  # 6
print(sol.maxStability(3, [[0,1,1,1],[1,2,1,1],[2,0,1,1]], 0))  # -1
